### ЗАДАЧА: Панель взыскания дебиторской задолженности по паттерну `MVC`

Команда finance operations работает с просроченными счетами и кейсами по взысканию дебиторской задолженности.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `invoice_id` — идентификатор счета;
- `client` — клиент;
- `invoice_amount` — исходная сумма счета;
- `paid_amount` — сумма, уже оплаченная клиентом;
- `days_overdue` — количество дней просрочки;
- `penalty_rate` — ставка штрафа за 30 дней просрочки;
- `penalty_amount` — начисленный штраф;
- `total_due` — остаток ко взысканию;
- `status` — текущий статус кейса;
- `collector` — сотрудник, который ведет кейс;
- `payment_plan_months` — срок плана погашения в месяцах;
- `decision` — итоговое решение.

### Формулы

Для каждого кейса при создании и после платежей нужно правильно считать:
- `penalty_amount = invoice_amount * penalty_rate * days_overdue / 30`, если `days_overdue > 0`, иначе `0.0`;
- `total_due = invoice_amount + penalty_amount - paid_amount`;
- оба значения нужно округлять до 2 знаков.

Также нужна сводка:
- количество кейсов по статусам;
- `total_open_due` — общая сумма `total_due` по кейсам, которые еще не закрыты и не эскалированы;
- `total_penalty` — общая сумма штрафов по всем кейсам;
- `total_paid` — общая сумма оплат по всем кейсам;
- `overdue_cases` — количество кейсов, где `days_overdue > 0`.

### Статусы

- `new`
- `in_collection`
- `plan_prepared`
- `ready_to_close`
- `closed`
- `escalated`

### Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `collector` несуществующему кейсу;
- финальные кейсы (`closed`, `escalated`) нельзя менять дальше;
- начать взыскание можно только из `new` и только если назначен `collector`;
- применить платеж можно только из `in_collection` или `plan_prepared`;
- платеж должен быть строго больше 0;
- после платежа нужно пересчитать `penalty_amount` и `total_due`;
- подготовить план погашения можно только из `in_collection`;
- срок плана должен быть от 1 до 12 месяцев;
- перевод в `ready_to_close` возможен только из `in_collection` или `plan_prepared`;
- перевод в `ready_to_close` возможен только если `abs(total_due) <= 0.01`;
- закрыть кейс можно только из `ready_to_close`;
- эскалировать кейс можно только из `in_collection` или `plan_prepared`.

In [3]:
from dataclasses import dataclass


initial_cases = [
    ("IC-100", "INV-5001", "Acme LLC", 12000.0, 2000.0, 45, 0.06),
    ("IC-101", "INV-5002", "Bravo Trade", 8000.0, 8000.0, 0, 0.05),
]

actions = [
    ("show",),
    ("collect", "IC-100"),
    ("assign", "IC-100", "Olga"),
    ("collect", "IC-100"),
    ("plan", "IC-100", 6),
    ("payment", "IC-100", 5000.0),
    ("ready", "IC-100"),
    ("payment", "IC-100", 6080.0),
    ("ready", "IC-100"),
    ("close", "IC-100", "fully_paid"),
    ("create", "IC-102", "INV-5003", "City Market", 4000.0, 0.0, 20, 0.05),
    ("assign", "IC-102", "Max"),
    ("collect", "IC-102"),
    ("escalate", "IC-102"),
    ("show",),
]


@dataclass
class CollectionCase:
    case_id: str
    invoice_id: str
    client: str
    invoice_amount: float
    paid_amount: float
    days_overdue: int
    penalty_rate: float
    penalty_amount: float
    total_due: float
    status: str = "new"
    collector: str | None = None
    payment_plan_months: int | None = None
    decision: str | None = None


class CollectionModel:
    final_statuses = {'closed', 'escalated'}
    def __init__(self):
        self.cases = {}

    def _calculate_penalty(self, invoice_amount: float, penalty_rate: float, days_overdue: int) -> float:
        if days_overdue <= 0:
            return 0.0
        return round(invoice_amount * penalty_rate * days_overdue / 30, 2)

    def _calculate_total_due(self, invoice_amount: float, penalty_amount: float, paid_amount: float) -> float:
        return round(invoice_amount + penalty_amount - paid_amount, 2)
    
    def _get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError('Case not found')
        return self.cases[case_id]
    
    def add_case(self, case_id: str, invoice_id: str, client: str, invoice_amount: float, paid_amount: float, days_overdue: int, penalty_rate: float) -> None:
        if case_id in self.cases:
            raise ValueError('Case already exists')
        penalty_amount = self._calculate_penalty(invoice_amount, penalty_rate, days_overdue)
        total_due = self._calculate_total_due(invoice_amount, penalty_amount, paid_amount)
        self.cases[case_id] = CollectionCase(case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate, penalty_amount, total_due)

    def assign_collector(self, case_id: str, collector: str) -> None:
        case = self._get_case(case_id)
        case.collector = collector

    def start_collection(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status in self.final_statuses:
            raise ValueError('Cannot change cases with final statuses')
        if case.status != 'new' or case.collector is None:
            raise ValueError('Collector is required')
        case.status = "in_collection"

    def apply_payment(self, case_id: str, payment_amount: float) -> None:
        case = self._get_case(case_id)
        if case.status not in {'in_collection', 'plan_prepared'}:
            raise ValueError('incorrect status for apply_payment')
        case.paid_amount += payment_amount
        case.penalty_amount = self._calculate_penalty(case.invoice_amount, case.penalty_rate, case.days_overdue)
        case.total_due = self._calculate_total_due(case.invoice_amount, case.penalty_amount, case.paid_amount)

    def prepare_plan(self, case_id: str, months: int) -> None:
        case = self._get_case(case_id)
        if case.status != 'in_collection':
            raise ValueError('invalid status for prepare_plan')
        if months < 1 or months > 12:
            raise ValueError('invalid months value')
        case.payment_plan_months = months
        case.status = "plan_prepared"

    def mark_ready_to_close(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status not in {'in_connection', 'plan_prepared'}:
            raise ValueError('invalid status for mark_ready_to_close')
        if abs(case.total_due) > 0.01:
            raise ValueError('total_due must be <= 0.01')
        case.status = "ready_to_close"

    def close_case(self, case_id: str, decision: str) -> None:
        case = self._get_case(case_id)
        if case.status != 'ready_to_close':
            raise ValueError('Can close the case only if status is "ready_to_close"')
        case.status = "closed"
        case.decision = decision

    def escalate_case(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status not in {'in_collection', 'plan_prepared'}:
            raise ValueError('invalid status for escalate the case')
        case.status = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.invoice_id} | {case.client} | {case.invoice_amount} | {case.paid_amount} | {case.days_overdue} | "
                f"{case.penalty_amount} | {case.total_due} | {case.status} | {case.collector} | {case.payment_plan_months} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_cases": 0,
            "total_open_due": 0.0,
            "total_penalty": 0.0,
            "total_paid": 0.0,
            "overdue_cases": 0,
        }
        for case in self.cases.values():
            result[case.status] = result.get(case.status, 0) + 1
            result["total_cases"] += 1
            result["total_penalty"] += case.penalty_amount
            result["total_paid"] += case.paid_amount
            if case.status in {"closed", "escalated"}:
                result["total_open_due"] += case.total_due
            if case.days_overdue > 0:
                result["overdue_cases"] += 1
        return result


class CollectionView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Collection cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class CollectionController:
    def __init__(self, model: CollectionModel, view: CollectionView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, invoice_id: str, client: str, invoice_amount: float, paid_amount: float, days_overdue: int, penalty_rate: float) -> None:
        try:
            self.model.add_case(case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_collector(self, case_id: str, collector: str) -> None:
        try:
            self.model.assign_collector(case_id, collector)
            self.view.render_success(f"Collector assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_collection(self, case_id: str) -> None:
        try:
            self.model.start_collection(case_id)
            self.view.render_success(f"Case {case_id} moved to collection")
        except ValueError as error:
            self.view.render_error(str(error))

    def apply_payment(self, case_id: str, payment_amount: float) -> None:
        try:
            self.model.apply_payment(case_id, payment_amount)
            self.view.render_success(f"Payment applied to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def prepare_plan(self, case_id: str, months: int) -> None:
        try:
            self.model.prepare_plan(case_id, months)
            self.view.render_success(f"Plan prepared for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_to_close(self, case_id: str) -> None:
        try:
            self.model.mark_ready_to_close(case_id)
            self.view.render_success(f"Case {case_id} is ready to close")
        except ValueError as error:
            self.view.render_error(str(error))

    def close_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.close_case(case_id, decision)
            self.view.render_success(f"Case {case_id} closed")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())


model = CollectionModel()
view = CollectionView()
controller = CollectionController(model, view)

for case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate in initial_cases:
    model.add_case(case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate = action
        controller.create_case(case_id, invoice_id, client, invoice_amount, paid_amount, days_overdue, penalty_rate)
    elif action[0] == "assign":
        _, case_id, collector = action
        controller.assign_collector(case_id, collector)
    elif action[0] == "collect":
        _, case_id = action
        controller.start_collection(case_id)
    elif action[0] == "payment":
        _, case_id, payment_amount = action
        controller.apply_payment(case_id, payment_amount)
    elif action[0] == "plan":
        _, case_id, months = action
        controller.prepare_plan(case_id, months)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_to_close(case_id)
    elif action[0] == "close":
        _, case_id, decision = action
        controller.close_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


Collection cases:
IC-100 | INV-5001 | Acme LLC | 12000.0 | 2000.0 | 45 | 1080.0 | 11080.0 | new | None | None | None
IC-101 | INV-5002 | Bravo Trade | 8000.0 | 8000.0 | 0 | 0.0 | 0.0 | new | None | None | None
Summary: {'total_cases': 2, 'total_open_due': 0.0, 'total_penalty': 1080.0, 'total_paid': 10000.0, 'overdue_cases': 1, 'new': 2}
ERROR: Collector is required
SUCCESS: Collector assigned to IC-100
SUCCESS: Case IC-100 moved to collection
SUCCESS: Plan prepared for IC-100
SUCCESS: Payment applied to IC-100
ERROR: total_due must be <= 0.01
SUCCESS: Payment applied to IC-100
SUCCESS: Case IC-100 is ready to close
SUCCESS: Case IC-100 closed
SUCCESS: Case IC-102 created
SUCCESS: Collector assigned to IC-102
SUCCESS: Case IC-102 moved to collection
SUCCESS: Case IC-102 escalated
Collection cases:
IC-100 | INV-5001 | Acme LLC | 12000.0 | 13080.0 | 45 | 1080.0 | 0.0 | closed | Olga | 6 | fully_paid
IC-101 | INV-5002 | Bravo Trade | 8000.0 | 8000.0 | 0 | 0.0 | 0.0 | new | None | None | No

In [ ]:
# Collection cases:
# IC-100 | INV-5001 | Acme LLC | 12000.0 | 2000.0 | 45 | 1080.0 | 11080.0 | new | None | None | None
# IC-101 | INV-5002 | Bravo Trade | 8000.0 | 8000.0 | 0 | 0.0 | 0.0 | new | None | None | None
# Summary: {'total_cases': 2, 'total_open_due': 11080.0, 'total_penalty': 1080.0, 'total_paid': 10000.0, 'overdue_cases': 1, 'new': 2}
# ERROR: Collector is required
# SUCCESS: Collector assigned to IC-100
# SUCCESS: Case IC-100 moved to collection
# SUCCESS: Plan prepared for IC-100
# SUCCESS: Payment applied to IC-100
# ERROR: Outstanding amount is not balanced
# SUCCESS: Payment applied to IC-100
# SUCCESS: Case IC-100 is ready to close
# SUCCESS: Case IC-100 closed
# SUCCESS: Case IC-102 created
# SUCCESS: Collector assigned to IC-102
# SUCCESS: Case IC-102 moved to collection
# SUCCESS: Case IC-102 escalated
# Collection cases:
# IC-100 | INV-5001 | Acme LLC | 12000.0 | 13080.0 | 45 | 1080.0 | 0.0 | closed | Olga | 6 | fully_paid
# IC-101 | INV-5002 | Bravo Trade | 8000.0 | 8000.0 | 0 | 0.0 | 0.0 | new | None | None | None
# IC-102 | INV-5003 | City Market | 4000.0 | 0.0 | 20 | 133.33 | 4133.33 | escalated | Max | None | escalated
# Summary: {'total_cases': 3, 'total_open_due': 0.0, 'total_penalty': 1213.33, 'total_paid': 21080.0, 'overdue_cases': 2, 'closed': 1, 'new': 1, 'escalated': 1}
#
# Финальное состояние:
# Collection cases:
# IC-100 | INV-5001 | Acme LLC | 12000.0 | 13080.0 | 45 | 1080.0 | 0.0 | closed | Olga | 6 | fully_paid
# IC-101 | INV-5002 | Bravo Trade | 8000.0 | 8000.0 | 0 | 0.0 | 0.0 | new | None | None | None
# IC-102 | INV-5003 | City Market | 4000.0 | 0.0 | 20 | 133.33 | 4133.33 | escalated | Max | None | escalated
# Summary: {'total_cases': 3, 'total_open_due': 0.0, 'total_penalty': 1213.33, 'total_paid': 21080.0, 'overdue_cases': 2, 'closed': 1, 'new': 1, 'escalated': 1}